# LSTM Sentiment Analysis — IMDB

Shared plumbing (tokenizer, vocab, padding, embedding, mask, data split, eval, training loop) is carried over from `RNN/rnn_sentiment.ipynb`. The LSTM-specific pieces (gates, cell state, params, forward) are built below, step by step.

In [10]:
import re
from collections import Counter

from datasets import load_dataset

In [11]:
def tokenize(text):
    # lowercase, then pull out runs of word chars -> strips punctuation/HTML tags for free
    return re.findall(r"\b\w+\b", text.lower())

In [12]:
def build_vocab(texts, min_freq=2):
    '''
    This function takes whole text as input, tokenizes it and outputs a
    dict, that has the token as the key and the tokenid as the value.
    
    '''
    counts = Counter()
    for text in texts:
        counts.update(tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counts.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

In [13]:
def encode(text, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in tokenize(text)]

In [14]:
imdb = load_dataset("stanfordnlp/imdb")
print(imdb)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [15]:
sample = imdb["train"][0]
print("label:", sample["label"], "(0 = neg, 1 = pos)")
print("text:", sample["text"][:300], "...")

label: 0 (0 = neg, 1 = pos)
text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...


In [16]:
vocab = build_vocab(imdb["train"]["text"])
print("vocab size:", len(vocab))

vocab size: 47010


In [17]:
encode(sample["text"], vocab)[:20]

[2, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 18]

In [18]:
import torch

In [19]:
def pad_batch(encoded_reviews, pad_value=0):
    '''
    Right-pads a batch of variable-length encoded reviews to the batch's own max length.

    Returns:
        padded: (batch_size, max_len) torch.long tensor of token ids
        lengths: (batch_size,) torch.long tensor of each review's real (unpadded) length
    '''
    lengths = torch.tensor([len(r) for r in encoded_reviews], dtype=torch.long)
    max_len = lengths.max().item()

    padded = torch.full((len(encoded_reviews), max_len), pad_value, dtype=torch.long)
    for i, review in enumerate(encoded_reviews):
        padded[i, :len(review)] = torch.tensor(review, dtype=torch.long)

    return padded, lengths

In [20]:
test_reviews = [imdb["train"][i]["text"] for i in range(3)]
test_encoded = [encode(text, vocab) for text in test_reviews]
print("lengths before padding:", [len(r) for r in test_encoded])

padded, lengths = pad_batch(test_encoded)
print("padded shape:", padded.shape)
print("lengths:", lengths)
print(padded[:, :15])  # first 15 columns of each row, to eyeball the padding

lengths before padding: [297, 228, 97]
padded shape: torch.Size([3, 297])
lengths: tensor([297, 228,  97])
tensor([[  2,   3,   2,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,
          15],
        [  2,   4,   5,   6,  54,  42, 181,  91, 182, 183, 184,  18, 177, 178,
         185],
        [ 33, 207,  36, 291,  75,  38, 292,  12, 155,  23,  14, 293,  38, 155,
          54]])


In [21]:
embed_dim = 100
embedding = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_dim, padding_idx=vocab["<PAD>"])

In [22]:
embedded = embedding(padded)
print("padded shape:", padded.shape)
print("embedded shape:", embedded.shape)

padded shape: torch.Size([3, 297])
embedded shape: torch.Size([3, 297, 100])


In [23]:
max_len = padded.shape[1]
mask = (torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1))
print("mask shape:", mask.shape)
print("real tokens per row:", mask.sum(dim=1))

mask shape: torch.Size([3, 297])
real tokens per row: tensor([297, 228,  97])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

max_tokens = 400
n_train = 22000
n_val = 2000
bs = 128
epochs = 27
lr = 0.2

torch.manual_seed(0)
perm = torch.randperm(25000)
subset = imdb["train"].select(perm[:n_train].tolist())
val_subset = imdb["train"].select(perm[n_train:n_train + n_val].tolist())

encoded = [encode(t, vocab)[-max_tokens:] for t in subset["text"]]
labels = torch.tensor(subset["label"], dtype=torch.float32).unsqueeze(1)
val_encoded = [encode(t, vocab)[-max_tokens:] for t in val_subset["text"]]
val_labels = torch.tensor(val_subset["label"], dtype=torch.float32).unsqueeze(1)

print("train positive fraction:", labels.mean().item(), "| val positive fraction:", val_labels.mean().item())
print("texts shared between train and val:", len(set(subset["text"]) & set(val_subset["text"])))

embedding = torch.nn.Embedding(len(vocab), embed_dim, padding_idx=vocab["<PAD>"]).to(device)

device: cuda
train positive fraction: 0.5018181800842285 | val positive fraction: 0.47850000858306885
texts shared between train and val: 12


## LSTM cell — built step by step

_(gates, cell state `c_t`, hidden state `h_t`, params, `forward` go here)_

In [75]:
hidden_dim = 128

W_xf = (torch.randn(embed_dim, hidden_dim, device=device) * 0.01).requires_grad_()
W_hf = (torch.randn(hidden_dim, hidden_dim, device=device) / hidden_dim ** 0.5).requires_grad_()
b_f = torch.ones(hidden_dim, device=device, requires_grad=True)  # sigmoid(1)~0.73: start mostly remembering

print("W_xf:", W_xf.shape, "W_hf:", W_hf.shape, "b_f:", b_f.shape)
print("leaf?", W_xf.is_leaf, W_hf.is_leaf, b_f.is_leaf)

W_xf: torch.Size([100, 128]) W_hf: torch.Size([128, 128]) b_f: torch.Size([128])
leaf? True True True


In [76]:
batch_size = padded.shape[0]
h = torch.zeros(batch_size, hidden_dim, device=device)
x_t = embedded[:, 0, :].to(device)

f = torch.sigmoid(x_t @ W_xf + h @ W_hf + b_f)
print("f shape:", f.shape)
print("f range:", f.min().item(), "to", f.max().item())

f shape: torch.Size([3, 128])
f range: 0.6701868772506714 to 0.7768391370773315


In [77]:
W_xi = (torch.randn(embed_dim, hidden_dim, device=device) * 0.01).requires_grad_()
W_hi = (torch.randn(hidden_dim, hidden_dim, device=device) / hidden_dim ** 0.5).requires_grad_()
b_i = torch.zeros(hidden_dim, device=device, requires_grad=True)

W_xg = (torch.randn(embed_dim, hidden_dim, device=device) * 0.01).requires_grad_()
W_hg = (torch.randn(hidden_dim, hidden_dim, device=device) / hidden_dim ** 0.5).requires_grad_()
b_g = torch.zeros(hidden_dim, device=device, requires_grad=True)

print("i params:", W_xi.shape, W_hi.shape, b_i.shape, "| g params:", W_xg.shape, W_hg.shape, b_g.shape)
print("leaf?", all(t.is_leaf for t in [W_xi, W_hi, b_i, W_xg, W_hg, b_g]))

i params: torch.Size([100, 128]) torch.Size([128, 128]) torch.Size([128]) | g params: torch.Size([100, 128]) torch.Size([128, 128]) torch.Size([128])
leaf? True


In [78]:
i = torch.sigmoid(x_t @ W_xi + h @ W_hi + b_i)
g = torch.tanh(x_t @ W_xg + h @ W_hg + b_g)
print("i shape:", i.shape, "| range:", i.min().item(), "to", i.max().item())
print("g shape:", g.shape, "| range:", g.min().item(), "to", g.max().item())

i shape: torch.Size([3, 128]) | range: 0.4303138256072998 to 0.5631227493286133
g shape: torch.Size([3, 128]) | range: -0.3804647624492645 to 0.3576901853084564


In [79]:
c = torch.zeros(batch_size, hidden_dim, device=device)

c = f * c + i * g
print("c shape:", c.shape)

c shape: torch.Size([3, 128])


In [80]:
W_xo = (torch.randn(embed_dim, hidden_dim, device=device) * 0.01).requires_grad_()
W_ho = (torch.randn(hidden_dim, hidden_dim, device=device) / hidden_dim ** 0.5).requires_grad_()
b_o = torch.zeros(hidden_dim, device=device, requires_grad=True)

print("o params:", W_xo.shape, W_ho.shape, b_o.shape, "| leaf?", all(t.is_leaf for t in [W_xo, W_ho, b_o]))

o params: torch.Size([100, 128]) torch.Size([128, 128]) torch.Size([128]) | leaf? True


In [81]:
o = torch.sigmoid(x_t @ W_xo + h @ W_ho + b_o)
print("o shape:", o.shape, "| range:", o.min().item(), "to", o.max().item())

o shape: torch.Size([3, 128]) | range: 0.39271238446235657 to 0.5991954207420349


In [82]:
h = o * torch.tanh(c)
print("h shape:", h.shape)

h shape: torch.Size([3, 128])


In [83]:
h = torch.zeros(batch_size, hidden_dim, device=device)
c = torch.zeros(batch_size, hidden_dim, device=device)
mask_d = mask.to(device)

for t in range(max_len):
    x_t = embedded[:, t, :].to(device)
    m = mask_d[:, t].unsqueeze(1)

    f = torch.sigmoid(x_t @ W_xf + h @ W_hf + b_f)
    i = torch.sigmoid(x_t @ W_xi + h @ W_hi + b_i)
    g = torch.tanh(x_t @ W_xg + h @ W_hg + b_g)
    o = torch.sigmoid(x_t @ W_xo + h @ W_ho + b_o)

    c_new = f * c + i * g
    h_new = o * torch.tanh(c_new)

    c = torch.where(m, c_new, c)
    h = torch.where(m, h_new, h)

print("final h:", h.shape, "| final c:", c.shape)

final h: torch.Size([3, 128]) | final c: torch.Size([3, 128])


In [84]:
def forward(padded, lengths, training=False, p_drop=0.3):
    B, T = padded.shape
    embedded = embedding(padded)
    embedded = torch.nn.functional.dropout(embedded, p=p_drop, training=training)
    mask = torch.arange(T, device=padded.device).unsqueeze(0) < lengths.unsqueeze(1)

    h = torch.zeros(B, hidden_dim, device=padded.device)
    c = torch.zeros(B, hidden_dim, device=padded.device)
    for t in range(T):
        x_t = embedded[:, t, :]
        m = mask[:, t].unsqueeze(1)

        f = torch.sigmoid(x_t @ W_xf + h @ W_hf + b_f)
        i = torch.sigmoid(x_t @ W_xi + h @ W_hi + b_i)
        g = torch.tanh(x_t @ W_xg + h @ W_hg + b_g)
        o = torch.sigmoid(x_t @ W_xo + h @ W_ho + b_o)

        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)

        c = torch.where(m, c_new, c)
        h = torch.where(m, h_new, h)

    return h @ W_hy + b_y

In [85]:
W_hy = (torch.randn(hidden_dim, 1, device=device) * 0.01).requires_grad_()
b_y = torch.zeros(1, device=device, requires_grad=True)

params = [embedding.weight,
          W_xf, W_hf, b_f,
          W_xi, W_hi, b_i,
          W_xg, W_hg, b_g,
          W_xo, W_ho, b_o,
          W_hy, b_y]
print("all leaf & on device:", all(p.is_leaf and p.device.type == device.type for p in params))
print(f"{sum(p.numel() for p in params):,} params")

all leaf & on device: True
4,818,377 params


In [86]:
def forward_fast(padded, lengths, training=False, p_drop=0.3):
    # same math as forward(), same 12 gate tensors: they're concatenated on the fly
    # (differentiable), so gradients still land on W_xf, W_hf, ... and forward() stays valid.
    B, T = padded.shape
    embedded = embedding(padded)
    embedded = torch.nn.functional.dropout(embedded, p=p_drop, training=training)
    mask = torch.arange(T, device=padded.device).unsqueeze(0) < lengths.unsqueeze(1)

    W_x = torch.cat([W_xf, W_xi, W_xg, W_xo], dim=1)   # (embed_dim, 4*hidden), gate order f, i, g, o
    W_h = torch.cat([W_hf, W_hi, W_hg, W_ho], dim=1)   # (hidden, 4*hidden)
    b = torch.cat([b_f, b_i, b_g, b_o])                # (4*hidden,)

    x_steps = (embedded @ W_x + b).unbind(dim=1)       # (B, T, 4*hidden) split along time: tuple of T tensors, each (B, 4*hidden)

    h = torch.zeros(B, hidden_dim, device=padded.device)
    c = torch.zeros(B, hidden_dim, device=padded.device)
    for t in range(T):
        m = mask[:, t].unsqueeze(1)

        z = x_steps[t] + h @ W_h                  # only the recurrent matmul stays in the loop
        f, i, g, o = z.chunk(4, dim=1)
        f, i, o = torch.sigmoid(f), torch.sigmoid(i), torch.sigmoid(o)
        g = torch.tanh(g)

        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)

        c = torch.where(m, c_new, c)
        h = torch.where(m, h_new, h)

    return h @ W_hy + b_y

In [87]:
# forward_fast must match forward (eval mode, no dropout)
with torch.no_grad():
    a = forward(padded.to(device), lengths.to(device))
    b_ = forward_fast(padded.to(device), lengths.to(device))
print("max abs diff:", (a - b_).abs().max().item(), "| allclose:", torch.allclose(a, b_, atol=1e-5))

max abs diff: 4.6566128730773926e-09 | allclose: True


In [88]:
def evaluate(encoded_reviews, labels, bs=128):
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(encoded_reviews), bs):
            padded_b, lengths_b = pad_batch(encoded_reviews[start:start + bs])
            padded_b, lengths_b = padded_b.to(device), lengths_b.to(device)
            y_b = labels[start:start + bs].to(device)

            logits = forward_fast(padded_b, lengths_b)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_b)
            total_loss += loss.item() * len(y_b)
            total_correct += ((logits > 0).float() == y_b).sum().item()
    return total_loss / len(encoded_reviews), total_correct / len(encoded_reviews)

In [90]:
from tqdm.auto import tqdm
lr = 0.05
epochs = 15
for epoch in range(epochs):
    order = torch.randperm(n_train)
    total_loss, total_correct = 0.0, 0
    for start in tqdm(range(0, n_train, bs), desc=f"epoch {epoch}"):
        idx = order[start:start + bs].tolist()
        padded_b, lengths_b = pad_batch([encoded[i] for i in idx])
        padded_b, lengths_b = padded_b.to(device), lengths_b.to(device)
        y_b = labels[idx].to(device)

        logits = forward_fast(padded_b, lengths_b, training=True)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_b)

        for p in params:
            p.grad = None
        loss.backward()
        with torch.no_grad():
            for p in params:
                p -= lr * p.grad

        total_loss += loss.item() * len(idx)
        total_correct += ((logits > 0).float() == y_b).sum().item()

    val_loss, val_acc = evaluate(val_encoded, val_labels)
    print(f"epoch {epoch}: train loss {total_loss / n_train:.4f} acc {total_correct / n_train:.3f} | val loss {val_loss:.4f} acc {val_acc:.3f}")

epoch 0:   0%|          | 0/172 [00:00<?, ?it/s]

epoch 0: train loss 0.3765 acc 0.825 | val loss 0.3888 acc 0.823


epoch 1:   0%|          | 0/172 [00:00<?, ?it/s]

epoch 1: train loss 0.3706 acc 0.829 | val loss 0.3859 acc 0.825


epoch 2:   0%|          | 0/172 [00:00<?, ?it/s]

...

epoch 7: train loss 0.3488 acc 0.843 | val loss 0.4001 acc 0.818


epoch 8:   0%|          | 0/172 [00:00<?, ?it/s]

epoch 8: train loss 0.3494 acc 0.841 | val loss 0.4041 acc 0.822


epoch 9:   0%|          | 0/172 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [96]:
# Held-out test set: 25k reviews the model never saw. Run once, after all tuning is done.
test_encoded = [encode(t, vocab)[-max_tokens:] for t in imdb["test"]["text"]]
test_labels = torch.tensor(imdb["test"]["label"], dtype=torch.float32).unsqueeze(1)

test_loss, test_acc = evaluate(test_encoded, test_labels)
print(f"test: {len(test_encoded)} reviews | loss {test_loss:.4f} | acc {test_acc:.4f}")

test: 25000 reviews | loss 0.3634 | acc 0.8343


In [95]:
checkpoint = {
    # weights, in the same order as `params`
    "params": [p.detach().cpu() for p in params],
    "param_names": ["embedding.weight",
                    "W_xf", "W_hf", "b_f",
                    "W_xi", "W_hi", "b_i",
                    "W_xg", "W_hg", "b_g",
                    "W_xo", "W_ho", "b_o",
                    "W_hy", "b_y"],

    # without these the weights are meaningless
    "vocab": vocab,              # row i of the embedding table means whatever THIS vocab says
    "max_tokens": max_tokens,    # inputs must be truncated the same way (last max_tokens tokens)
    "embed_dim": embed_dim,      # to rebuild the Embedding module
    "hidden_dim": hidden_dim,    # forward() reads this as a global
    "readout": "final_h",        # no pooling: logits come from the last real timestep's h

    # provenance
    "val_acc": val_acc,
    "val_loss": val_loss,
    "test_acc": test_acc,
    "test_loss": test_loss,
}

torch.save(checkpoint, "lstm_imdb.pt")
print(f"saved lstm_imdb.pt | {sum(p.numel() for p in params):,} params | test acc {test_acc:.4f}")

saved lstm_imdb.pt | 4,818,377 params | test acc 0.8343
